## FILE_FIX: corrección de rutas de files.csv

Compara cada ruta de `files.csv` con los ficheros físicos de la carpeta exportada de la clínica. Cuando la ruta no coincide exactamente, busca el fichero más parecido dentro de la carpeta del cliente (similitud mínima del 93 %, con `SequenceMatcher`) y corrige la ruta.

Genera cuatro salidas: el `files.csv` corregido, un CSV de auditoría con la ruta original y la corregida, la lista de ficheros que no se encuentran en las carpetas y la lista de duplicados eliminados.

In [ ]:
#!/usr/bin/env python3
import os
import csv
import io
import re
import logging
import unicodedata
from pathlib import Path, PurePosixPath
from difflib import SequenceMatcher
from collections import defaultdict

# ==========================================================
# LOGGING
# ==========================================================
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("files_corrector")

# ==========================================================
# CONFIG
# ==========================================================
METADATA_CSV_PATH = Path(r"C:\Migracion\ClinicaX\Destino\files.csv")
FOLDER_FILE_PATH = Path(r"C:\Migracion\ClinicaX\Export\ficheros")
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")

OUTPUT_MAIN_NAME = "files.csv"
AUDIT_NAME = "files.csv_corrected.csv"
MISSING_NAME = "files_fisicos_no_encontrados_en_carpetas.csv"
DUPLICATES_NAME = "files_duplicados_eliminados.csv"

MIN_RATIO = 0.93
PROGRESS_EVERY = 100
OUTPUT_ENCODING = "utf-8-sig"

FILEPATH_COLUMN_PREFERRED = "FilePath"
FILENAME_COLUMN_PREFERRED = "FileName"
FILECLIENTID_COLUMN_PREFERRED = "FileClientID"

AUDIT_CORRECTED_COL = "FilePath_corrected"

MISSING_REASON_COL = "MissingReason"
MISSING_BESTSCORE_COL = "BestScore"
MISSING_EXPECTEDDIR_COL = "ExpectedDir"

DUPLICATE_REASON_COL = "DuplicateReason"


# =============================
# Helpers
# =============================
def normalize_header(s: str) -> str:
    if s is None:
        return ""
    return s.strip().lstrip("\ufeff")


def normalize_headers_list(headers):
    return [normalize_header(h) for h in (headers or [])]


def find_column(headers, preferred_name: str):
    if not headers:
        return None

    preferred_norm = normalize_header(preferred_name)

    for h in headers:
        if normalize_header(h) == preferred_norm:
            return normalize_header(h)

    preferred_lower = preferred_norm.lower()
    for h in headers:
        if normalize_header(h).lower() == preferred_lower:
            return normalize_header(h)

    return None


def detect_encoding(path: Path) -> str:
    raw = path.read_bytes()[:16384]

    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"

    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            continue

    return "latin1"


def sniff_dialect_and_encoding(path: Path):
    raw = path.read_bytes()[:8192]
    sample = raw.decode("latin1", errors="replace")

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,|\t,")
    except csv.Error:
        dialect = csv.get_dialect("excel")

    return dialect, detect_encoding(path)


def open_csv_skipping_blank_header_lines(path: Path, encoding: str):
    text = path.read_text(encoding=encoding, errors="replace")
    lines = text.splitlines(True)

    i = 0
    while i < len(lines) and lines[i].strip() == "":
        i += 1

    trimmed = "".join(lines[i:])
    if trimmed.strip() == "":
        raise ValueError(f"El CSV parece vacío o solo contiene líneas en blanco: {path}")

    return io.StringIO(trimmed)


def preview_first_nonempty_lines(path: Path, encoding: str, n: int = 5) -> str:
    text = path.read_text(encoding=encoding, errors="replace")
    out = []

    for line in text.splitlines():
        if line.strip():
            out.append(line)
        if len(out) >= n:
            break

    return "\n".join(out)


def normalize_path_str(p: str) -> str:
    p = (p or "").strip().replace("\\", "/")
    p = re.sub(r"/{2,}", "/", p)
    return p


def filename_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""
    return PurePosixPath(fp).name


def expected_dir_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""

    parent = str(PurePosixPath(fp).parent).replace("\\", "/")
    if parent == ".":
        return ""
    return parent


def build_corrected_path(dir_part: str, filename: str) -> str:
    dir_part = normalize_path_str(dir_part)
    filename = (filename or "").strip()

    if dir_part and filename:
        return normalize_path_str(f"{dir_part}/{filename}")
    if filename:
        return normalize_path_str(filename)
    return dir_part


def physical_path_from_metadata(fp_norm: str, folder_root: Path) -> Path:
    fp_norm = normalize_path_str(fp_norm)
    p = Path(fp_norm)

    if p.is_absolute():
        return p

    base_dir = folder_root.resolve().parent
    return (base_dir / Path(fp_norm)).resolve()


# =============================
# Texto / mojibake / comparación
# =============================
def mojibake_score(s: str) -> int:
    bad_markers = ("Ã", "Â", "ï¿½", "�")
    return sum(s.count(x) for x in bad_markers)


def maybe_fix_mojibake(s: str) -> str:
    s = (s or "").strip()
    best = s
    best_score = mojibake_score(s)

    for src_enc in ("latin1", "cp1252"):
        try:
            candidate = s.encode(src_enc).decode("utf-8")
            candidate_score = mojibake_score(candidate)
            if candidate_score < best_score:
                best = candidate
                best_score = candidate_score
        except Exception:
            pass

    return best


def clean_text_for_output(s: str) -> str:
    s = (s or "").strip()
    s = maybe_fix_mojibake(s)
    return s


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))


def compare_key(name: str) -> str:
    s = clean_text_for_output(name)
    s = strip_accents(s)
    s = s.replace(" .", ".").replace(". ", ".")
    s = s.replace(" ", "")
    s = s.replace("�", "")
    s = s.lower()
    return s


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, compare_key(a), compare_key(b)).ratio()


# =============================
# Índice realidad física
# =============================
def build_index(files_root: Path):
    index_by_dir = defaultdict(list)
    keymap_by_dir = defaultdict(dict)
    collisions = defaultdict(set)

    files_root = files_root.resolve()
    root_name = files_root.name

    for dirpath, _, filenames in os.walk(str(files_root)):
        rel_inside = os.path.relpath(dirpath, str(files_root)).replace("\\", "/")
        dir_key = root_name if rel_inside == "." else f"{root_name}/{rel_inside}"
        dir_key = normalize_path_str(dir_key)

        for fn in filenames:
            fn_clean = clean_text_for_output(fn)
            index_by_dir[dir_key].append(fn_clean)

            k = compare_key(fn_clean)
            if k in keymap_by_dir[dir_key]:
                collisions[dir_key].add(k)
            else:
                keymap_by_dir[dir_key][k] = fn_clean

    for d, ks in collisions.items():
        for k in ks:
            keymap_by_dir[d].pop(k, None)

    return index_by_dir, keymap_by_dir


def find_best_match(dir_rel: str, target_filename: str, index_by_dir, keymap_by_dir, min_ratio: float):
    dir_rel = normalize_path_str(dir_rel)
    target_filename = clean_text_for_output(target_filename)

    k = compare_key(target_filename)
    direct = keymap_by_dir.get(dir_rel, {}).get(k)
    if direct:
        return direct, 1.0

    candidates = index_by_dir.get(dir_rel, [])
    if not candidates:
        return None, 0.0

    target_k = compare_key(target_filename)
    best_fn, best_score = None, 0.0

    for real_fn in candidates:
        score = similarity(target_k, compare_key(real_fn))
        if score > best_score:
            best_score = score
            best_fn = real_fn

    if best_score >= min_ratio:
        return best_fn, best_score

    return None, best_score


# =============================
# Duplicados por cliente
# =============================
def remove_duplicates_by_client(rows, client_col: str, filename_col: str):
    seen = defaultdict(set)
    unique_rows = []
    duplicate_rows = []

    for row in rows:
        row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
        client_id = clean_text_for_output(row_norm.get(client_col, ""))
        filename = clean_text_for_output(row_norm.get(filename_col, ""))

        dedupe_key = compare_key(filename)

        if dedupe_key not in seen[client_id]:
            seen[client_id].add(dedupe_key)
            unique_rows.append(row)
        else:
            duplicate_row = {k: row_norm.get(k, "") for k in row_norm.keys()}
            duplicate_row[DUPLICATE_REASON_COL] = "DUPLICATED_FILENAME_FOR_SAME_CLIENT"
            duplicate_rows.append(duplicate_row)

    return unique_rows, duplicate_rows


def sanitize_row_for_output(row: dict, fieldnames: list[str]) -> dict:
    out = {}
    for k in fieldnames:
        v = row.get(k, "")
        if isinstance(v, str):
            v = clean_text_for_output(v)
            if "Path" in k:
                v = normalize_path_str(v)
        out[k] = v
    return out


def run():
    if not METADATA_CSV_PATH.is_file():
        raise FileNotFoundError(f"No existe el CSV de entrada: {METADATA_CSV_PATH}")
    if not FOLDER_FILE_PATH.is_dir():
        raise NotADirectoryError(f"No existe la carpeta de archivos: {FOLDER_FILE_PATH}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    output_main_csv = OUTPUT_DIR / OUTPUT_MAIN_NAME
    output_audit_csv = OUTPUT_DIR / AUDIT_NAME
    output_missing_csv = OUTPUT_DIR / MISSING_NAME
    output_duplicates_csv = OUTPUT_DIR / DUPLICATES_NAME

    tmp_audit_csv = OUTPUT_DIR / (AUDIT_NAME + ".tmp")
    tmp_missing_csv = OUTPUT_DIR / (MISSING_NAME + ".tmp")
    tmp_duplicates_csv = OUTPUT_DIR / (DUPLICATES_NAME + ".tmp")

    dialect, encoding = sniff_dialect_and_encoding(METADATA_CSV_PATH)
    delim = getattr(dialect, "delimiter", ",")
    logger.info(f"CSV detectado: delimiter='{delim}' encoding='{encoding}'")

    logger.info(f"Indexando REALIDAD en (recursivo): {FOLDER_FILE_PATH} ...")
    index_by_dir, keymap_by_dir = build_index(FOLDER_FILE_PATH)
    logger.info(f"Indexado completado. Directorios: {len(index_by_dir)}")

    ok = fixed = not_found = duplicates_removed = 0
    processed = 0
    written_main = 0

    csv_stream = open_csv_skipping_blank_header_lines(METADATA_CSV_PATH, encoding)

    with open(str(output_main_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_main, \
         open(str(tmp_audit_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_audit, \
         open(str(tmp_missing_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_missing, \
         open(str(tmp_duplicates_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_duplicates:

        reader = csv.DictReader(csv_stream, dialect=dialect)

        raw_fieldnames = list(reader.fieldnames or [])
        norm_fieldnames = normalize_headers_list(raw_fieldnames)

        filepath_col = find_column(norm_fieldnames, FILEPATH_COLUMN_PREFERRED)
        filename_col = find_column(norm_fieldnames, FILENAME_COLUMN_PREFERRED)
        clientid_col = find_column(norm_fieldnames, FILECLIENTID_COLUMN_PREFERRED)

        if not filepath_col:
            logger.error("No se encontró la columna de ruta en el CSV.")
            logger.error(f"Headers detectados (raw): {raw_fieldnames}")
            logger.error(f"Headers detectados (norm): {norm_fieldnames}")
            logger.error("\n" + preview_first_nonempty_lines(METADATA_CSV_PATH, encoding, n=5))
            raise KeyError(f"El CSV no tiene columna '{FILEPATH_COLUMN_PREFERRED}' (o header raro).")

        if not filename_col:
            raise KeyError(f"El CSV no tiene columna '{FILENAME_COLUMN_PREFERRED}'.")

        if not clientid_col:
            raise KeyError(f"El CSV no tiene columna '{FILECLIENTID_COLUMN_PREFERRED}'.")

        main_fieldnames = list(norm_fieldnames)

        audit_fieldnames = list(norm_fieldnames)
        if AUDIT_CORRECTED_COL not in audit_fieldnames:
            audit_fieldnames.append(AUDIT_CORRECTED_COL)

        missing_fieldnames = list(norm_fieldnames)
        for extra in (MISSING_REASON_COL, MISSING_BESTSCORE_COL, MISSING_EXPECTEDDIR_COL):
            if extra not in missing_fieldnames:
                missing_fieldnames.append(extra)

        duplicates_fieldnames = list(norm_fieldnames)
        if DUPLICATE_REASON_COL not in duplicates_fieldnames:
            duplicates_fieldnames.append(DUPLICATE_REASON_COL)

        main_writer = csv.DictWriter(f_main, fieldnames=main_fieldnames, dialect=dialect)
        audit_writer = csv.DictWriter(f_audit, fieldnames=audit_fieldnames, dialect=dialect)
        missing_writer = csv.DictWriter(f_missing, fieldnames=missing_fieldnames, dialect=dialect)
        duplicates_writer = csv.DictWriter(f_duplicates, fieldnames=duplicates_fieldnames, dialect=dialect)

        main_writer.writeheader()
        audit_writer.writeheader()
        missing_writer.writeheader()
        duplicates_writer.writeheader()

        rows = list(reader)
        total_rows_input = len(rows)

        unique_rows, duplicate_rows = remove_duplicates_by_client(rows, clientid_col, filename_col)
        total_unique_after_duplicates = len(unique_rows)

        for dup_row in duplicate_rows:
            dup_out = sanitize_row_for_output(dup_row, duplicates_fieldnames)
            duplicates_writer.writerow(dup_out)
            duplicates_removed += 1

        for row in unique_rows:
            processed += 1
            if PROGRESS_EVERY and processed % PROGRESS_EVERY == 0:
                logger.info(f"Procesadas {processed} filas...")

            row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
            row_norm = {k: clean_text_for_output(v) if isinstance(v, str) else v for k, v in row_norm.items()}

            original_fp = normalize_path_str(row_norm.get(filepath_col, ""))
            original_name = clean_text_for_output(row_norm.get(filename_col, ""))

            main_row_out = {k: row_norm.get(k, "") for k in main_fieldnames}
            main_row_out = sanitize_row_for_output(main_row_out, main_fieldnames)

            if not original_fp:
                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            dir_part = expected_dir_from_filepath(original_fp)
            path_filename = filename_from_filepath(original_fp)

            physical = physical_path_from_metadata(original_fp, FOLDER_FILE_PATH)
            if physical.is_file():
                real_name = clean_text_for_output(physical.name)
                corrected = build_corrected_path(dir_part, real_name)

                name_changed = compare_key(real_name) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = real_name

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = real_name
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            candidates_to_try = []
            if original_name:
                candidates_to_try.append(original_name)
            if path_filename and compare_key(path_filename) != compare_key(original_name):
                candidates_to_try.append(path_filename)

            best = None
            score = 0.0
            for candidate_name in candidates_to_try:
                candidate_best, candidate_score = find_best_match(
                    dir_part,
                    candidate_name,
                    index_by_dir,
                    keymap_by_dir,
                    MIN_RATIO
                )
                if candidate_best and candidate_score >= score:
                    best = candidate_best
                    score = candidate_score

            if best:
                corrected = build_corrected_path(dir_part, best)

                name_changed = compare_key(best) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = best

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = best
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
            else:
                not_found += 1
                missing_row = {k: row_norm.get(k, "") for k in norm_fieldnames}
                missing_row[MISSING_REASON_COL] = "NOT_FOUND_IN_PHYSICAL_FOLDERS"
                missing_row[MISSING_BESTSCORE_COL] = f"{score:.3f}"
                missing_row[MISSING_EXPECTEDDIR_COL] = dir_part
                missing_row = sanitize_row_for_output(missing_row, missing_fieldnames)
                missing_writer.writerow(missing_row)

    def finalize_optional_csv(tmp_path: Path, final_path: Path, count: int):
        try:
            if count > 0:
                if final_path.exists():
                    final_path.unlink()
                tmp_path.replace(final_path)
            else:
                if tmp_path.exists():
                    tmp_path.unlink()
        except Exception as e:
            logger.error(f"No se pudo finalizar '{final_path.name}': {e}")
            raise

    finalize_optional_csv(tmp_audit_csv, output_audit_csv, fixed)
    finalize_optional_csv(tmp_missing_csv, output_missing_csv, not_found)
    finalize_optional_csv(tmp_duplicates_csv, output_duplicates_csv, duplicates_removed)

    logger.info("-" * 60)
    logger.info(f"TOTAL leídas origen:         {total_rows_input}")
    logger.info(f"TOTAL únicas tras duplicados:{total_unique_after_duplicates}")
    logger.info(f"TOTAL registros finales CSV: {written_main}")
    logger.info(f"OK (ya existían):            {ok}")
    logger.info(f"CORREGIDOS:                  {fixed}")
    logger.info(f"DUPLICADOS ELIMINADOS:       {duplicates_removed}")
    logger.info(f"NO ENCONTRADOS:              {not_found}")
    logger.info(f"Salida normal:               {output_main_csv}")

    if fixed > 0:
        logger.info(f"Corregidos (CSV):            {output_audit_csv}")
    else:
        logger.info("Corregidos (CSV):            no generado (fixed=0)")

    if duplicates_removed > 0:
        logger.info(f"Duplicados eliminados (CSV): {output_duplicates_csv}")
    else:
        logger.info("Duplicados eliminados (CSV): no generado (duplicates_removed=0)")

    if not_found > 0:
        logger.info(f"Missing (CSV):               {output_missing_csv}")
    else:
        logger.info("Missing (CSV):               no generado (not_found=0)")

    logger.info("FIN")


run()

## PICTURE_FIX: corrección de rutas de pictures.csv

Compara cada ruta de `pictures.csv` con los ficheros físicos de la carpeta exportada de la clínica. Cuando la ruta no coincide exactamente, busca el fichero más parecido dentro de la carpeta del cliente (similitud mínima del 93 %, con `SequenceMatcher`) y corrige la ruta.

Genera cuatro salidas: el `pictures.csv` corregido, un CSV de auditoría con la ruta original y la corregida, la lista de ficheros que no se encuentran en las carpetas y la lista de duplicados eliminados.

In [ ]:
#!/usr/bin/env python3
import os
import csv
import io
import re
import logging
import unicodedata
from pathlib import Path, PurePosixPath
from difflib import SequenceMatcher
from collections import defaultdict

# ==========================================================
# LOGGING
# ==========================================================
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("pictures_corrector")

# ==========================================================
# CONFIG
# ==========================================================
METADATA_CSV_PATH = Path(r"C:\Migracion\ClinicaX\Destino\pictures.csv")
FOLDER_FILE_PATH = Path(r"C:\Migracion\ClinicaX\Export\ficheros")
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")

OUTPUT_MAIN_NAME = "pictures.csv"
AUDIT_NAME = "pictures.csv_corrected.csv"
MISSING_NAME = "pictures_fisicos_no_encontrados_en_carpetas.csv"
DUPLICATES_NAME = "pictures_duplicados_eliminados.csv"

MIN_RATIO = 0.93
PROGRESS_EVERY = 100
OUTPUT_ENCODING = "utf-8-sig"

FILEPATH_COLUMN_PREFERRED = "PicturePath"
PICTURECOMMENTS_COLUMN_PREFERRED = "PictureComments"
PICTURECLIENTID_COLUMN_PREFERRED = "PictureClientID"

AUDIT_CORRECTED_COL = "PicturePath_corrected"

MISSING_REASON_COL = "MissingReason"
MISSING_BESTSCORE_COL = "BestScore"
MISSING_EXPECTEDDIR_COL = "ExpectedDir"

DUPLICATE_REASON_COL = "DuplicateReason"


# =============================
# Helpers
# =============================
def normalize_header(s: str) -> str:
    if s is None:
        return ""
    return s.strip().lstrip("\ufeff")


def normalize_headers_list(headers):
    return [normalize_header(h) for h in (headers or [])]


def find_column(headers, preferred_name: str):
    if not headers:
        return None

    preferred_norm = normalize_header(preferred_name)

    for h in headers:
        if normalize_header(h) == preferred_norm:
            return normalize_header(h)

    preferred_lower = preferred_norm.lower()
    for h in headers:
        if normalize_header(h).lower() == preferred_lower:
            return normalize_header(h)

    return None


def detect_encoding(path: Path) -> str:
    raw = path.read_bytes()[:16384]

    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"

    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            continue

    return "latin1"


def sniff_dialect_and_encoding(path: Path):
    raw = path.read_bytes()[:8192]
    sample = raw.decode("latin1", errors="replace")

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,|\t,")
    except csv.Error:
        dialect = csv.get_dialect("excel")

    return dialect, detect_encoding(path)


def open_csv_skipping_blank_header_lines(path: Path, encoding: str):
    text = path.read_text(encoding=encoding, errors="replace")
    lines = text.splitlines(True)

    i = 0
    while i < len(lines) and lines[i].strip() == "":
        i += 1

    trimmed = "".join(lines[i:])
    if trimmed.strip() == "":
        raise ValueError(f"El CSV parece vacío o solo contiene líneas en blanco: {path}")

    return io.StringIO(trimmed)


def preview_first_nonempty_lines(path: Path, encoding: str, n: int = 5) -> str:
    text = path.read_text(encoding=encoding, errors="replace")
    out = []

    for line in text.splitlines():
        if line.strip():
            out.append(line)
        if len(out) >= n:
            break

    return "\n".join(out)


def normalize_path_str(p: str) -> str:
    p = (p or "").strip().replace("\\", "/")
    p = re.sub(r"/{2,}", "/", p)
    return p


def filename_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""
    return PurePosixPath(fp).name


def expected_dir_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""

    parent = str(PurePosixPath(fp).parent).replace("\\", "/")
    if parent == ".":
        return ""
    return parent


def build_corrected_path(dir_part: str, filename: str) -> str:
    dir_part = normalize_path_str(dir_part)
    filename = (filename or "").strip()

    if dir_part and filename:
        return normalize_path_str(f"{dir_part}/{filename}")
    if filename:
        return normalize_path_str(filename)
    return dir_part


def physical_path_from_metadata(fp_norm: str, folder_root: Path) -> Path:
    fp_norm = normalize_path_str(fp_norm)
    p = Path(fp_norm)

    if p.is_absolute():
        return p

    base_dir = folder_root.resolve().parent
    return (base_dir / Path(fp_norm)).resolve()


# =============================
# Texto / mojibake / comparación
# =============================
def mojibake_score(s: str) -> int:
    bad_markers = ("Ã", "Â", "ï¿½", "�")
    return sum(s.count(x) for x in bad_markers)


def maybe_fix_mojibake(s: str) -> str:
    s = (s or "").strip()
    best = s
    best_score = mojibake_score(s)

    for src_enc in ("latin1", "cp1252"):
        try:
            candidate = s.encode(src_enc).decode("utf-8")
            candidate_score = mojibake_score(candidate)
            if candidate_score < best_score:
                best = candidate
                best_score = candidate_score
        except Exception:
            pass

    return best


def clean_text_for_output(s: str) -> str:
    s = (s or "").strip()
    s = maybe_fix_mojibake(s)
    return s


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))


def compare_key(name: str) -> str:
    s = clean_text_for_output(name)
    s = strip_accents(s)
    s = s.replace(" .", ".").replace(". ", ".")
    s = s.replace(" ", "")
    s = s.replace("�", "")
    s = s.lower()
    return s


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, compare_key(a), compare_key(b)).ratio()


# =============================
# Índice realidad física
# =============================
def build_index(files_root: Path):
    index_by_dir = defaultdict(list)
    keymap_by_dir = defaultdict(dict)
    collisions = defaultdict(set)

    files_root = files_root.resolve()
    root_name = files_root.name

    for dirpath, _, filenames in os.walk(str(files_root)):
        rel_inside = os.path.relpath(dirpath, str(files_root)).replace("\\", "/")
        dir_key = root_name if rel_inside == "." else f"{root_name}/{rel_inside}"
        dir_key = normalize_path_str(dir_key)

        for fn in filenames:
            fn_clean = clean_text_for_output(fn)
            index_by_dir[dir_key].append(fn_clean)

            k = compare_key(fn_clean)
            if k in keymap_by_dir[dir_key]:
                collisions[dir_key].add(k)
            else:
                keymap_by_dir[dir_key][k] = fn_clean

    for d, ks in collisions.items():
        for k in ks:
            keymap_by_dir[d].pop(k, None)

    return index_by_dir, keymap_by_dir


def find_best_match(dir_rel: str, target_filename: str, index_by_dir, keymap_by_dir, min_ratio: float):
    dir_rel = normalize_path_str(dir_rel)
    target_filename = clean_text_for_output(target_filename)

    k = compare_key(target_filename)
    direct = keymap_by_dir.get(dir_rel, {}).get(k)
    if direct:
        return direct, 1.0

    candidates = index_by_dir.get(dir_rel, [])
    if not candidates:
        return None, 0.0

    target_k = compare_key(target_filename)
    best_fn, best_score = None, 0.0

    for real_fn in candidates:
        score = similarity(target_k, compare_key(real_fn))
        if score > best_score:
            best_score = score
            best_fn = real_fn

    if best_score >= min_ratio:
        return best_fn, best_score

    return None, best_score


# =============================
# Duplicados por cliente
# =============================
def remove_duplicates_by_client(rows, client_col: str, filename_col: str):
    seen = defaultdict(set)
    unique_rows = []
    duplicate_rows = []

    for row in rows:
        row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
        client_id = clean_text_for_output(row_norm.get(client_col, ""))
        filename = clean_text_for_output(row_norm.get(filename_col, ""))

        dedupe_key = compare_key(filename)

        if dedupe_key not in seen[client_id]:
            seen[client_id].add(dedupe_key)
            unique_rows.append(row)
        else:
            duplicate_row = {k: row_norm.get(k, "") for k in row_norm.keys()}
            duplicate_row[DUPLICATE_REASON_COL] = "DUPLICATED_FILENAME_FOR_SAME_CLIENT"
            duplicate_rows.append(duplicate_row)

    return unique_rows, duplicate_rows


def sanitize_row_for_output(row: dict, fieldnames: list[str]) -> dict:
    out = {}
    for k in fieldnames:
        v = row.get(k, "")
        if isinstance(v, str):
            v = clean_text_for_output(v)
            if "Path" in k:
                v = normalize_path_str(v)
        out[k] = v
    return out


def run():
    if not METADATA_CSV_PATH.is_file():
        raise FileNotFoundError(f"No existe el CSV de entrada: {METADATA_CSV_PATH}")
    if not FOLDER_FILE_PATH.is_dir():
        raise NotADirectoryError(f"No existe la carpeta de archivos: {FOLDER_FILE_PATH}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    output_main_csv = OUTPUT_DIR / OUTPUT_MAIN_NAME
    output_audit_csv = OUTPUT_DIR / AUDIT_NAME
    output_missing_csv = OUTPUT_DIR / MISSING_NAME
    output_duplicates_csv = OUTPUT_DIR / DUPLICATES_NAME

    tmp_audit_csv = OUTPUT_DIR / (AUDIT_NAME + ".tmp")
    tmp_missing_csv = OUTPUT_DIR / (MISSING_NAME + ".tmp")
    tmp_duplicates_csv = OUTPUT_DIR / (DUPLICATES_NAME + ".tmp")

    dialect, encoding = sniff_dialect_and_encoding(METADATA_CSV_PATH)
    delim = getattr(dialect, "delimiter", ",")
    logger.info(f"CSV detectado: delimiter='{delim}' encoding='{encoding}'")

    logger.info(f"Indexando REALIDAD en (recursivo): {FOLDER_FILE_PATH} ...")
    index_by_dir, keymap_by_dir = build_index(FOLDER_FILE_PATH)
    logger.info(f"Indexado completado. Directorios: {len(index_by_dir)}")

    ok = fixed = not_found = duplicates_removed = 0
    processed = 0
    written_main = 0

    csv_stream = open_csv_skipping_blank_header_lines(METADATA_CSV_PATH, encoding)

    with open(str(output_main_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_main, \
         open(str(tmp_audit_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_audit, \
         open(str(tmp_missing_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_missing, \
         open(str(tmp_duplicates_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_duplicates:

        reader = csv.DictReader(csv_stream, dialect=dialect)

        raw_fieldnames = list(reader.fieldnames or [])
        norm_fieldnames = normalize_headers_list(raw_fieldnames)

        filepath_col = find_column(norm_fieldnames, FILEPATH_COLUMN_PREFERRED)
        filename_col = find_column(norm_fieldnames, PICTURECOMMENTS_COLUMN_PREFERRED)
        clientid_col = find_column(norm_fieldnames, PICTURECLIENTID_COLUMN_PREFERRED)

        if not filepath_col:
            logger.error("No se encontró la columna de ruta en el CSV.")
            logger.error(f"Headers detectados (raw): {raw_fieldnames}")
            logger.error(f"Headers detectados (norm): {norm_fieldnames}")
            logger.error("\n" + preview_first_nonempty_lines(METADATA_CSV_PATH, encoding, n=5))
            raise KeyError(f"El CSV no tiene columna '{FILEPATH_COLUMN_PREFERRED}' (o header raro).")

        if not filename_col:
            raise KeyError(f"El CSV no tiene columna '{PICTURECOMMENTS_COLUMN_PREFERRED}'.")

        if not clientid_col:
            raise KeyError(f"El CSV no tiene columna '{PICTURECLIENTID_COLUMN_PREFERRED}'.")

        main_fieldnames = list(norm_fieldnames)

        audit_fieldnames = list(norm_fieldnames)
        if AUDIT_CORRECTED_COL not in audit_fieldnames:
            audit_fieldnames.append(AUDIT_CORRECTED_COL)

        missing_fieldnames = list(norm_fieldnames)
        for extra in (MISSING_REASON_COL, MISSING_BESTSCORE_COL, MISSING_EXPECTEDDIR_COL):
            if extra not in missing_fieldnames:
                missing_fieldnames.append(extra)

        duplicates_fieldnames = list(norm_fieldnames)
        if DUPLICATE_REASON_COL not in duplicates_fieldnames:
            duplicates_fieldnames.append(DUPLICATE_REASON_COL)

        main_writer = csv.DictWriter(f_main, fieldnames=main_fieldnames, dialect=dialect)
        audit_writer = csv.DictWriter(f_audit, fieldnames=audit_fieldnames, dialect=dialect)
        missing_writer = csv.DictWriter(f_missing, fieldnames=missing_fieldnames, dialect=dialect)
        duplicates_writer = csv.DictWriter(f_duplicates, fieldnames=duplicates_fieldnames, dialect=dialect)

        main_writer.writeheader()
        audit_writer.writeheader()
        missing_writer.writeheader()
        duplicates_writer.writeheader()

        rows = list(reader)
        total_rows_input = len(rows)

        unique_rows, duplicate_rows = remove_duplicates_by_client(rows, clientid_col, filename_col)
        total_unique_after_duplicates = len(unique_rows)

        for dup_row in duplicate_rows:
            dup_out = sanitize_row_for_output(dup_row, duplicates_fieldnames)
            duplicates_writer.writerow(dup_out)
            duplicates_removed += 1

        for row in unique_rows:
            processed += 1
            if PROGRESS_EVERY and processed % PROGRESS_EVERY == 0:
                logger.info(f"Procesadas {processed} filas...")

            row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
            row_norm = {k: clean_text_for_output(v) if isinstance(v, str) else v for k, v in row_norm.items()}

            original_fp = normalize_path_str(row_norm.get(filepath_col, ""))
            original_name = clean_text_for_output(row_norm.get(filename_col, ""))

            main_row_out = {k: row_norm.get(k, "") for k in main_fieldnames}
            main_row_out = sanitize_row_for_output(main_row_out, main_fieldnames)

            if not original_fp:
                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            dir_part = expected_dir_from_filepath(original_fp)
            path_filename = filename_from_filepath(original_fp)

            physical = physical_path_from_metadata(original_fp, FOLDER_FILE_PATH)
            if physical.is_file():
                real_name = clean_text_for_output(physical.name)
                corrected = build_corrected_path(dir_part, real_name)

                name_changed = compare_key(real_name) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = real_name

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = real_name
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            candidates_to_try = []
            if original_name:
                candidates_to_try.append(original_name)
            if path_filename and compare_key(path_filename) != compare_key(original_name):
                candidates_to_try.append(path_filename)

            best = None
            score = 0.0
            for candidate_name in candidates_to_try:
                candidate_best, candidate_score = find_best_match(
                    dir_part,
                    candidate_name,
                    index_by_dir,
                    keymap_by_dir,
                    MIN_RATIO
                )
                if candidate_best and candidate_score >= score:
                    best = candidate_best
                    score = candidate_score

            if best:
                corrected = build_corrected_path(dir_part, best)

                name_changed = compare_key(best) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = best

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = best
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
            else:
                not_found += 1
                missing_row = {k: row_norm.get(k, "") for k in norm_fieldnames}
                missing_row[MISSING_REASON_COL] = "NOT_FOUND_IN_PHYSICAL_FOLDERS"
                missing_row[MISSING_BESTSCORE_COL] = f"{score:.3f}"
                missing_row[MISSING_EXPECTEDDIR_COL] = dir_part
                missing_row = sanitize_row_for_output(missing_row, missing_fieldnames)
                missing_writer.writerow(missing_row)

    def finalize_optional_csv(tmp_path: Path, final_path: Path, count: int):
        try:
            if count > 0:
                if final_path.exists():
                    final_path.unlink()
                tmp_path.replace(final_path)
            else:
                if tmp_path.exists():
                    tmp_path.unlink()
        except Exception as e:
            logger.error(f"No se pudo finalizar '{final_path.name}': {e}")
            raise

    finalize_optional_csv(tmp_audit_csv, output_audit_csv, fixed)
    finalize_optional_csv(tmp_missing_csv, output_missing_csv, not_found)
    finalize_optional_csv(tmp_duplicates_csv, output_duplicates_csv, duplicates_removed)

    logger.info("-" * 60)
    logger.info(f"TOTAL leídas origen:         {total_rows_input}")
    logger.info(f"TOTAL únicas tras duplicados:{total_unique_after_duplicates}")
    logger.info(f"TOTAL registros finales CSV: {written_main}")
    logger.info(f"OK (ya existían):            {ok}")
    logger.info(f"CORREGIDOS:                  {fixed}")
    logger.info(f"DUPLICADOS ELIMINADOS:       {duplicates_removed}")
    logger.info(f"NO ENCONTRADOS:              {not_found}")
    logger.info(f"Salida normal:               {output_main_csv}")

    if fixed > 0:
        logger.info(f"Corregidos (CSV):            {output_audit_csv}")
    else:
        logger.info("Corregidos (CSV):            no generado (fixed=0)")

    if duplicates_removed > 0:
        logger.info(f"Duplicados eliminados (CSV): {output_duplicates_csv}")
    else:
        logger.info("Duplicados eliminados (CSV): no generado (duplicates_removed=0)")

    if not_found > 0:
        logger.info(f"Missing (CSV):               {output_missing_csv}")
    else:
        logger.info("Missing (CSV):               no generado (not_found=0)")

    logger.info("FIN")


run()

## POC: muestra de prueba

Aplica la misma corrección de rutas a `files.csv` y, además, genera `POC_files.csv` con los primeros 150 registros únicos y copia esos ficheros físicos a una carpeta `POC`. Lo usé para la carga de prueba antes de la migración completa.

In [ ]:
#!/usr/bin/env python3
import os
import csv
import io
import re
import logging
import unicodedata
import shutil  # Importar para copiar archivos físicos
from pathlib import Path, PurePosixPath
from difflib import SequenceMatcher
from collections import defaultdict

# ==========================================================
# LOGGING
# ==========================================================
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("files_corrector")

# ==========================================================
# CONFIG
# ==========================================================
METADATA_CSV_PATH = Path(r"C:\Migracion\ClinicaX\Destino\files.csv")
FOLDER_FILE_PATH = Path(r"C:\Migracion\ClinicaX\Export\ficheros")
OUTPUT_DIR = Path(r"C:\Migracion\ClinicaX\Destino")

OUTPUT_MAIN_NAME = "files.csv"
AUDIT_NAME = "files.csv_corrected.csv"
MISSING_NAME = "files_fisicos_no_encontrados_en_carpetas.csv"
DUPLICATES_NAME = "files_duplicados_eliminados.csv"

MIN_RATIO = 0.93
PROGRESS_EVERY = 100
OUTPUT_ENCODING = "utf-8-sig"
SIZE_POC = 150  # Número de registros para el archivo POC

FILEPATH_COLUMN_PREFERRED = "FilePath"
FILENAME_COLUMN_PREFERRED = "FileName"
FILECLIENTID_COLUMN_PREFERRED = "FileClientID"

AUDIT_CORRECTED_COL = "FilePath_corrected"
MISSING_REASON_COL = "MissingReason"
MISSING_BESTSCORE_COL = "BestScore"
MISSING_EXPECTEDDIR_COL = "ExpectedDir"
DUPLICATE_REASON_COL = "DuplicateReason"

# =============================
# Helpers
# =============================
def normalize_header(s: str) -> str:
    if s is None:
        return ""
    return s.strip().lstrip("\ufeff")


def normalize_headers_list(headers):
    return [normalize_header(h) for h in (headers or [])]


def find_column(headers, preferred_name: str):
    if not headers:
        return None

    preferred_norm = normalize_header(preferred_name)

    for h in headers:
        if normalize_header(h) == preferred_norm:
            return normalize_header(h)

    preferred_lower = preferred_norm.lower()
    for h in headers:
        if normalize_header(h).lower() == preferred_lower:
            return normalize_header(h)

    return None


def detect_encoding(path: Path) -> str:
    raw = path.read_bytes()[:16384]

    if raw.startswith(b"\xef\xbb\xbf"):
        return "utf-8-sig"

    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            raw.decode(enc)
            return enc
        except UnicodeDecodeError:
            continue

    return "latin1"


def sniff_dialect_and_encoding(path: Path):
    raw = path.read_bytes()[:8192]
    sample = raw.decode("latin1", errors="replace")

    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=";,|\t,")
    except csv.Error:
        dialect = csv.get_dialect("excel")

    return dialect, detect_encoding(path)


def open_csv_skipping_blank_header_lines(path: Path, encoding: str):
    text = path.read_text(encoding=encoding, errors="replace")
    lines = text.splitlines(True)

    i = 0
    while i < len(lines) and lines[i].strip() == "":
        i += 1

    trimmed = "".join(lines[i:])
    if trimmed.strip() == "":
        raise ValueError(f"El CSV parece vacío o solo contiene líneas en blanco: {path}")

    return io.StringIO(trimmed)


def preview_first_nonempty_lines(path: Path, encoding: str, n: int = 5) -> str:
    text = path.read_text(encoding=encoding, errors="replace")
    out = []

    for line in text.splitlines():
        if line.strip():
            out.append(line)
        if len(out) >= n:
            break

    return "\n".join(out)


def normalize_path_str(p: str) -> str:
    p = (p or "").strip().replace("\\", "/")
    p = re.sub(r"/{2,}", "/", p)
    return p


def filename_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""
    return PurePosixPath(fp).name


def expected_dir_from_filepath(fp: str) -> str:
    fp = normalize_path_str(fp)
    if not fp:
        return ""

    parent = str(PurePosixPath(fp).parent).replace("\\", "/")
    if parent == ".":
        return ""
    return parent


def build_corrected_path(dir_part: str, filename: str) -> str:
    dir_part = normalize_path_str(dir_part)
    filename = (filename or "").strip()

    if dir_part and filename:
        return normalize_path_str(f"{dir_part}/{filename}")
    if filename:
        return normalize_path_str(filename)
    return dir_part


def physical_path_from_metadata(fp_norm: str, folder_root: Path) -> Path:
    fp_norm = normalize_path_str(fp_norm)
    p = Path(fp_norm)

    if p.is_absolute():
        return p

    base_dir = folder_root.resolve().parent
    return (base_dir / Path(fp_norm)).resolve()


# =============================
# Texto / mojibake / comparación
# =============================
def mojibake_score(s: str) -> int:
    bad_markers = ("Ã", "Â", "ï¿½", "�")
    return sum(s.count(x) for x in bad_markers)


def maybe_fix_mojibake(s: str) -> str:
    s = (s or "").strip()
    best = s
    best_score = mojibake_score(s)

    for src_enc in ("latin1", "cp1252"):
        try:
            candidate = s.encode(src_enc).decode("utf-8")
            candidate_score = mojibake_score(candidate)
            if candidate_score < best_score:
                best = candidate
                best_score = candidate_score
        except Exception:
            pass

    return best


def clean_text_for_output(s: str) -> str:
    s = (s or "").strip()
    s = maybe_fix_mojibake(s)
    return s


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFKD", s)
    return "".join(ch for ch in s if not unicodedata.combining(ch))


def compare_key(name: str) -> str:
    s = clean_text_for_output(name)
    s = strip_accents(s)
    s = s.replace(" .", ".").replace(". ", ".")
    s = s.replace(" ", "")
    s = s.replace("�", "")
    s = s.lower()
    return s


def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, compare_key(a), compare_key(b)).ratio()


# =============================
# Índice realidad física
# =============================
def build_index(files_root: Path):
    index_by_dir = defaultdict(list)
    keymap_by_dir = defaultdict(dict)
    collisions = defaultdict(set)

    files_root = files_root.resolve()
    root_name = files_root.name

    for dirpath, _, filenames in os.walk(str(files_root)):
        rel_inside = os.path.relpath(dirpath, str(files_root)).replace("\\", "/")
        dir_key = root_name if rel_inside == "." else f"{root_name}/{rel_inside}"
        dir_key = normalize_path_str(dir_key)

        for fn in filenames:
            fn_clean = clean_text_for_output(fn)
            index_by_dir[dir_key].append(fn_clean)

            k = compare_key(fn_clean)
            if k in keymap_by_dir[dir_key]:
                collisions[dir_key].add(k)
            else:
                keymap_by_dir[dir_key][k] = fn_clean

    for d, ks in collisions.items():
        for k in ks:
            keymap_by_dir[d].pop(k, None)

    return index_by_dir, keymap_by_dir


def find_best_match(dir_rel: str, target_filename: str, index_by_dir, keymap_by_dir, min_ratio: float):
    dir_rel = normalize_path_str(dir_rel)
    target_filename = clean_text_for_output(target_filename)

    k = compare_key(target_filename)
    direct = keymap_by_dir.get(dir_rel, {}).get(k)
    if direct:
        return direct, 1.0

    candidates = index_by_dir.get(dir_rel, [])
    if not candidates:
        return None, 0.0

    target_k = compare_key(target_filename)
    best_fn, best_score = None, 0.0

    for real_fn in candidates:
        score = similarity(target_k, compare_key(real_fn))
        if score > best_score:
            best_score = score
            best_fn = real_fn

    if best_score >= min_ratio:
        return best_fn, best_score

    return None, best_score


# =============================
# Duplicados por cliente
# =============================
def remove_duplicates_by_client(rows, client_col: str, filename_col: str):
    seen = defaultdict(set)
    unique_rows = []
    duplicate_rows = []

    for row in rows:
        row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
        client_id = clean_text_for_output(row_norm.get(client_col, ""))
        filename = clean_text_for_output(row_norm.get(filename_col, ""))

        dedupe_key = compare_key(filename)

        if dedupe_key not in seen[client_id]:
            seen[client_id].add(dedupe_key)
            unique_rows.append(row)
        else:
            duplicate_row = {k: row_norm.get(k, "") for k in row_norm.keys()}
            duplicate_row[DUPLICATE_REASON_COL] = "DUPLICATED_FILENAME_FOR_SAME_CLIENT"
            duplicate_rows.append(duplicate_row)

    return unique_rows, duplicate_rows


def sanitize_row_for_output(row: dict, fieldnames: list[str]) -> dict:
    out = {}
    for k in fieldnames:
        v = row.get(k, "")
        if isinstance(v, str):
            v = clean_text_for_output(v)
            if "Path" in k:
                v = normalize_path_str(v)
        out[k] = v
    return out


# =============================
# Run function con los ajustes
# =============================
def run():
    if not METADATA_CSV_PATH.is_file():
        raise FileNotFoundError(f"No existe el CSV de entrada: {METADATA_CSV_PATH}")
    if not FOLDER_FILE_PATH.is_dir():
        raise NotADirectoryError(f"No existe la carpeta de archivos: {FOLDER_FILE_PATH}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Crear carpeta POC si no existe
    poc_dir = OUTPUT_DIR / "POC"
    poc_dir.mkdir(parents=True, exist_ok=True)
    
    output_main_csv = OUTPUT_DIR / OUTPUT_MAIN_NAME
    output_audit_csv = OUTPUT_DIR / AUDIT_NAME
    output_missing_csv = OUTPUT_DIR / MISSING_NAME
    output_duplicates_csv = OUTPUT_DIR / DUPLICATES_NAME

    tmp_audit_csv = OUTPUT_DIR / (AUDIT_NAME + ".tmp")
    tmp_missing_csv = OUTPUT_DIR / (MISSING_NAME + ".tmp")
    tmp_duplicates_csv = OUTPUT_DIR / (DUPLICATES_NAME + ".tmp")

    dialect, encoding = sniff_dialect_and_encoding(METADATA_CSV_PATH)
    delim = getattr(dialect, "delimiter", ",")
    logger.info(f"CSV detectado: delimiter='{delim}' encoding='{encoding}'")

    logger.info(f"Indexando REALIDAD en (recursivo): {FOLDER_FILE_PATH} ...")
    index_by_dir, keymap_by_dir = build_index(FOLDER_FILE_PATH)
    logger.info(f"Indexado completado. Directorios: {len(index_by_dir)}")

    ok = fixed = not_found = duplicates_removed = 0
    processed = 0
    written_main = 0

    csv_stream = open_csv_skipping_blank_header_lines(METADATA_CSV_PATH, encoding)

    with open(str(output_main_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_main, \
         open(str(tmp_audit_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_audit, \
         open(str(tmp_missing_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_missing, \
         open(str(tmp_duplicates_csv), "w", encoding=OUTPUT_ENCODING, newline="") as f_duplicates:

        reader = csv.DictReader(csv_stream, dialect=dialect)

        raw_fieldnames = list(reader.fieldnames or [])
        norm_fieldnames = normalize_headers_list(raw_fieldnames)

        filepath_col = find_column(norm_fieldnames, FILEPATH_COLUMN_PREFERRED)
        filename_col = find_column(norm_fieldnames, FILENAME_COLUMN_PREFERRED)
        clientid_col = find_column(norm_fieldnames, FILECLIENTID_COLUMN_PREFERRED)

        if not filepath_col:
            logger.error("No se encontró la columna de ruta en el CSV.")
            logger.error(f"Headers detectados (raw): {raw_fieldnames}")
            logger.error(f"Headers detectados (norm): {norm_fieldnames}")
            logger.error("\n" + preview_first_nonempty_lines(METADATA_CSV_PATH, encoding, n=5))
            raise KeyError(f"El CSV no tiene columna '{FILEPATH_COLUMN_PREFERRED}' (o header raro).")

        if not filename_col:
            raise KeyError(f"El CSV no tiene columna '{FILENAME_COLUMN_PREFERRED}'.")

        if not clientid_col:
            raise KeyError(f"El CSV no tiene columna '{FILECLIENTID_COLUMN_PREFERRED}'.")

        main_fieldnames = list(norm_fieldnames)

        audit_fieldnames = list(norm_fieldnames)
        if AUDIT_CORRECTED_COL not in audit_fieldnames:
            audit_fieldnames.append(AUDIT_CORRECTED_COL)

        missing_fieldnames = list(norm_fieldnames)
        for extra in (MISSING_REASON_COL, MISSING_BESTSCORE_COL, MISSING_EXPECTEDDIR_COL):
            if extra not in missing_fieldnames:
                missing_fieldnames.append(extra)

        duplicates_fieldnames = list(norm_fieldnames)
        if DUPLICATE_REASON_COL not in duplicates_fieldnames:
            duplicates_fieldnames.append(DUPLICATE_REASON_COL)

        main_writer = csv.DictWriter(f_main, fieldnames=main_fieldnames, dialect=dialect)
        audit_writer = csv.DictWriter(f_audit, fieldnames=audit_fieldnames, dialect=dialect)
        missing_writer = csv.DictWriter(f_missing, fieldnames=missing_fieldnames, dialect=dialect)
        duplicates_writer = csv.DictWriter(f_duplicates, fieldnames=duplicates_fieldnames, dialect=dialect)

        main_writer.writeheader()
        audit_writer.writeheader()
        missing_writer.writeheader()
        duplicates_writer.writeheader()

        rows = list(reader)
        total_rows_input = len(rows)

        unique_rows, duplicate_rows = remove_duplicates_by_client(rows, clientid_col, filename_col)
        total_unique_after_duplicates = len(unique_rows)

        for dup_row in duplicate_rows:
            dup_out = sanitize_row_for_output(dup_row, duplicates_fieldnames)
            duplicates_writer.writerow(dup_out)
            duplicates_removed += 1

        for row in unique_rows:
            processed += 1
            if PROGRESS_EVERY and processed % PROGRESS_EVERY == 0:
                logger.info(f"Procesadas {processed} filas...")

            row_norm = {normalize_header(k): v for k, v in (row or {}).items()}
            row_norm = {k: clean_text_for_output(v) if isinstance(v, str) else v for k, v in row_norm.items()}

            original_fp = normalize_path_str(row_norm.get(filepath_col, ""))
            original_name = clean_text_for_output(row_norm.get(filename_col, ""))

            main_row_out = {k: row_norm.get(k, "") for k in main_fieldnames}
            main_row_out = sanitize_row_for_output(main_row_out, main_fieldnames)

            if not original_fp:
                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            dir_part = expected_dir_from_filepath(original_fp)
            path_filename = filename_from_filepath(original_fp)

            physical = physical_path_from_metadata(original_fp, FOLDER_FILE_PATH)
            if physical.is_file():
                real_name = clean_text_for_output(physical.name)
                corrected = build_corrected_path(dir_part, real_name)

                name_changed = compare_key(real_name) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = real_name

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = real_name
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
                continue

            candidates_to_try = []
            if original_name:
                candidates_to_try.append(original_name)
            if path_filename and compare_key(path_filename) != compare_key(original_name):
                candidates_to_try.append(path_filename)

            best = None
            score = 0.0
            for candidate_name in candidates_to_try:
                candidate_best, candidate_score = find_best_match(
                    dir_part,
                    candidate_name,
                    index_by_dir,
                    keymap_by_dir,
                    MIN_RATIO
                )
                if candidate_best and candidate_score >= score:
                    best = candidate_best
                    score = candidate_score

            if best:
                corrected = build_corrected_path(dir_part, best)

                name_changed = compare_key(best) != compare_key(original_name)
                path_changed = normalize_path_str(corrected) != normalize_path_str(original_fp)

                if name_changed or path_changed:
                    main_row_out[filepath_col] = corrected
                    main_row_out[filename_col] = best

                    audit_row_out = {k: row_norm.get(k, "") for k in norm_fieldnames}
                    audit_row_out[filename_col] = best
                    audit_row_out[AUDIT_CORRECTED_COL] = corrected
                    audit_row_out = sanitize_row_for_output(audit_row_out, audit_fieldnames)
                    audit_writer.writerow(audit_row_out)
                    fixed += 1
                else:
                    ok += 1

                main_writer.writerow(main_row_out)
                written_main += 1
            else:
                not_found += 1
                missing_row = {k: row_norm.get(k, "") for k in norm_fieldnames}
                missing_row[MISSING_REASON_COL] = "NOT_FOUND_IN_PHYSICAL_FOLDERS"
                missing_row[MISSING_BESTSCORE_COL] = f"{score:.3f}"
                missing_row[MISSING_EXPECTEDDIR_COL] = dir_part
                missing_row = sanitize_row_for_output(missing_row, missing_fieldnames)
                missing_writer.writerow(missing_row)

    # Renombrar archivos .tmp a archivos finales
    def finalize_optional_csv(tmp_path: Path, final_path: Path, count: int):
        try:
            if count > 0:
                if final_path.exists():
                    final_path.unlink()
                tmp_path.replace(final_path)
            else:
                if tmp_path.exists():
                    tmp_path.unlink()
        except Exception as e:
            logger.error(f"No se pudo finalizar '{final_path.name}': {e}")
            raise

    finalize_optional_csv(tmp_audit_csv, output_audit_csv, fixed)
    finalize_optional_csv(tmp_missing_csv, output_missing_csv, not_found)
    finalize_optional_csv(tmp_duplicates_csv, output_duplicates_csv, duplicates_removed)

    # Generar archivo POC_files.csv en la misma carpeta que files.csv
    poc_file_path = OUTPUT_DIR / "POC_files.csv"
    with open(poc_file_path, "w", encoding=OUTPUT_ENCODING, newline="") as f_poc:
        poc_writer = csv.DictWriter(f_poc, fieldnames=main_fieldnames, dialect=dialect)
        poc_writer.writeheader()
        for row in unique_rows[:SIZE_POC]:
            row_out = sanitize_row_for_output(row, main_fieldnames)
            poc_writer.writerow(row_out)

    logger.info(f"Archivo POC generado: {poc_file_path}")

    # Copiar archivos físicos listados en POC_files.csv a la carpeta POC
    with open(poc_file_path, "r", encoding=OUTPUT_ENCODING) as f_poc:
        poc_reader = csv.DictReader(f_poc)
        for row in poc_reader:
            filepath = row.get(FILEPATH_COLUMN_PREFERRED, "")
            if filepath:
                physical_path = Path(filepath)
                if physical_path.exists():
                    destination = poc_dir / physical_path.name
                    shutil.copy(physical_path, destination)
                    logger.info(f"Archivo copiado a POC: {destination}")

    logger.info("-" * 60)
    logger.info(f"TOTAL leídas origen:         {total_rows_input}")
    logger.info(f"TOTAL únicas tras duplicados:{total_unique_after_duplicates}")
    logger.info(f"TOTAL registros finales CSV: {written_main}")
    logger.info(f"OK (ya existían):            {ok}")
    logger.info(f"CORREGIDOS:                  {fixed}")
    logger.info(f"DUPLICADOS ELIMINADOS:       {duplicates_removed}")
    logger.info(f"NO ENCONTRADOS:              {not_found}")
    logger.info(f"Salida normal:               {output_main_csv}")

    if fixed > 0:
        logger.info(f"Corregidos (CSV):            {output_audit_csv}")
    else:
        logger.info("Corregidos (CSV):            no generado (fixed=0)")

    if duplicates_removed > 0:
        logger.info(f"Duplicados eliminados (CSV): {output_duplicates_csv}")
    else:
        logger.info("Duplicados eliminados (CSV): no generado (duplicates_removed=0)")

    if not_found > 0:
        logger.info(f"Missing (CSV):               {output_missing_csv}")
    else:
        logger.info("Missing (CSV):               no generado (not_found=0)")

    logger.info("FIN")

run()